## Learning Outcomes

By the end of this lab, you should be able to:

- Measure re-identification risk through equivalence classes.
- Apply simple generalization and suppression to obtain k-anonymity [@sweeney2002kanonymity].
- Diagnose when k-anonymity still permits attribute disclosure.
- Check l-diversity [@machanavajjhala2006ldiversity] and t-closeness [@li2007tcloseness] on a sensitive attribute.
- Explain the privacy-utility tradeoff with model accuracy and information loss.
- Add differentially private noise to aggregate queries using sensitivity and `epsilon` [@dwork2006differential; @dwork2006calibrating].

We use the Adult income dataset because it is a classic public tabular dataset with demographic quasi-identifiers and a sensitive income label. If the dataset cannot be downloaded, the notebook falls back to a synthetic Adult-like dataset so the lab remains runnable offline.



## 1. Setup

If you run this lab in Google Colab or a minimal Python environment, run the following cell first.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = ["matplotlib", "numpy", "pandas", "sklearn"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if missing_packages:
    pip_names = ["scikit-learn" if pkg == "sklearn" else pkg for pkg in missing_packages]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pip_names])


In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
plt.style.use("seaborn-v0_8-whitegrid")


## 2. Load Adult or Create a Fallback


In [ ]:
def normalize_columns(df):
    df = df.copy()
    df.columns = [c.strip().lower().replace("-", "_") for c in df.columns]
    return df


def make_synthetic_adult(n=8000, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    age = rng.integers(18, 75, n)
    education_num = rng.choice(np.arange(1, 17), n, p=np.array([1, 1, 1, 1, 1, 2, 2, 3, 5, 7, 10, 13, 18, 18, 12, 5]) / 100)
    hours_per_week = np.clip(rng.normal(40, 12, n).round(), 1, 80).astype(int)
    sex = rng.choice(["Female", "Male"], n, p=[0.48, 0.52])
    race = rng.choice(["White", "Black", "Asian-Pac-Islander", "Amer-Indian-Eskimo", "Other"], n, p=[0.78, 0.10, 0.07, 0.02, 0.03])
    marital_status = rng.choice(["Never-married", "Married-civ-spouse", "Divorced", "Separated", "Widowed"], n, p=[0.34, 0.44, 0.12, 0.06, 0.04])
    workclass = rng.choice(["Private", "Self-emp", "Government", "Without-pay"], n, p=[0.74, 0.10, 0.15, 0.01])
    occupation = rng.choice(["Sales", "Craft-repair", "Exec-managerial", "Prof-specialty", "Service", "Admin-clerical"], n)
    native_country = rng.choice(["United-States", "Mexico", "Philippines", "Germany", "Canada", "Other"], n, p=[0.84, 0.05, 0.03, 0.02, 0.02, 0.04])

    logit = (
        -5.0
        + 0.045 * age
        + 0.28 * education_num
        + 0.025 * hours_per_week
        + 0.85 * (marital_status == "Married-civ-spouse")
        + 0.45 * np.isin(occupation, ["Exec-managerial", "Prof-specialty"])
        + 0.25 * (sex == "Male")
    )
    p_high_income = 1 / (1 + np.exp(-logit))
    income = np.where(rng.binomial(1, p_high_income), ">50K", "<=50K")

    return pd.DataFrame(
        {
            "age": age,
            "workclass": workclass,
            "education_num": education_num,
            "marital_status": marital_status,
            "occupation": occupation,
            "race": race,
            "sex": sex,
            "hours_per_week": hours_per_week,
            "native_country": native_country,
            "income": income,
        }
    )


def load_adult_sample(n=8000):
    try:
        adult = fetch_openml("adult", version=2, as_frame=True)
        df = normalize_columns(adult.frame)
        target_name = adult.target.name.lower().replace("-", "_")
        if "class" in df.columns and "income" not in df.columns:
            df = df.rename(columns={"class": "income"})
        elif target_name in df.columns and target_name != "income":
            df = df.rename(columns={target_name: "income"})
        df = df.replace("?", np.nan).dropna()
        df["income"] = df["income"].astype(str).str.replace(".", "", regex=False).str.strip()
        return df.sample(n=min(n, len(df)), random_state=RANDOM_STATE).reset_index(drop=True), "OpenML Adult"
    except Exception as exc:
        print(f"Using synthetic fallback because Adult could not be downloaded: {exc}")
        return make_synthetic_adult(n=n), "Synthetic Adult-like fallback"


df, data_source = load_adult_sample()
print(data_source)
display(df.head())
display(df["income"].value_counts(normalize=True).rename("income_share").round(3))


## 3. Define Quasi-Identifiers and a Sensitive Attribute

For this lab:

- Quasi-identifiers are demographic attributes that can help link a record to a person.
- The sensitive attribute is `income`, because the released table may reveal whether a person earns `>50K`.


In [ ]:
base_qi = ["age", "education_num", "marital_status", "race", "sex", "native_country"]
sensitive = "income"

df[base_qi + [sensitive]].head()


## 4. A Linkage Risk Baseline

The smallest equivalence class is the dataset's k-anonymity level for the chosen quasi-identifiers.


In [ ]:
def equivalence_class_sizes(data, qi_cols):
    return data.groupby(qi_cols, dropna=False).size().rename("equivalence_class_size")


def risk_report(data, qi_cols):
    sizes = equivalence_class_sizes(data, qi_cols)
    row_sizes = data.merge(sizes.reset_index(), on=qi_cols, how="left")["equivalence_class_size"]
    return pd.Series(
        {
            "records": len(data),
            "equivalence_classes": len(sizes),
            "k_min": int(sizes.min()),
            "share_unique": float((row_sizes == 1).mean()),
            "share_in_classes_lt_5": float((row_sizes < 5).mean()),
            "median_class_size": float(row_sizes.median()),
        }
    )


baseline_risk = risk_report(df, base_qi)
baseline_risk.round(3)


### Exercise

Interpret `share_unique`. If an attacker knows a person's exact values for the selected quasi-identifiers, what does this number mean?



## 5. Generalization

Generalization replaces precise values with coarser values. Here we use age bins, education bins, broad marital status, and broad country groups.


In [ ]:
def country_region(value):
    value = str(value)
    if value == "United-States":
        return "US"
    if value in {"Canada", "Mexico"}:
        return "North America other"
    if value in {"England", "Germany", "France", "Italy", "Portugal", "Poland", "Greece", "Ireland", "Scotland", "Hungary", "Yugoslavia", "Holand-Netherlands"}:
        return "Europe"
    if value in {"China", "Japan", "India", "Iran", "Philippines", "Vietnam", "Taiwan", "Thailand", "Cambodia", "Laos", "Hong", "South"}:
        return "Asia"
    if value in {"Cuba", "Jamaica", "Haiti", "Dominican-Republic", "Puerto-Rico", "Columbia", "Ecuador", "El-Salvador", "Guatemala", "Honduras", "Nicaragua", "Peru", "Trinadad&Tobago"}:
        return "Latin America / Caribbean"
    return "Other"


def generalize_adult(data, age_width=10, education_width=4, country="region", marital="broad"):
    out = data.copy()
    age_floor = (out["age"].astype(int) // age_width) * age_width
    out["age_gen"] = age_floor.astype(str) + "-" + (age_floor + age_width - 1).astype(str)

    edu_floor = ((out["education_num"].astype(int) - 1) // education_width) * education_width + 1
    out["education_gen"] = edu_floor.astype(str) + "-" + np.minimum(edu_floor + education_width - 1, 16).astype(str)

    if marital == "broad":
        out["marital_gen"] = np.where(out["marital_status"].astype(str).str.startswith("Married"), "Married", "Not married")
    else:
        out["marital_gen"] = out["marital_status"].astype(str)

    out["race_gen"] = out["race"].astype(str)
    out["sex_gen"] = out["sex"].astype(str)
    out["country_gen"] = out["native_country"].map(country_region) if country == "region" else out["native_country"].astype(str)
    return out


gen_qi = ["age_gen", "education_gen", "marital_gen", "race_gen", "sex_gen", "country_gen"]
release = generalize_adult(df, age_width=10, education_width=4, country="region", marital="broad")
display(release[gen_qi + [sensitive]].head())
display(risk_report(release, gen_qi).round(3))


## 6. Search for a k-Anonymous Release

This simple search is not an optimal anonymization algorithm. It is a transparent lab exercise that shows how stronger generalization increases anonymity and decreases detail.


In [ ]:
settings = []
for age_width in [5, 10, 15, 20]:
    for education_width in [2, 4, 8, 16]:
        for country in ["exact", "region"]:
            for marital in ["exact", "broad"]:
                candidate = generalize_adult(
                    df,
                    age_width=age_width,
                    education_width=education_width,
                    country=country,
                    marital=marital,
                )
                report = risk_report(candidate, gen_qi)
                settings.append(
                    {
                        "age_width": age_width,
                        "education_width": education_width,
                        "country": country,
                        "marital": marital,
                        **report.to_dict(),
                    }
                )

settings_df = pd.DataFrame(settings).sort_values(["k_min", "share_unique"], ascending=[False, True])
display(settings_df.head(10).round(3))


In [ ]:
target_k = 5
k_candidates = settings_df[settings_df["k_min"] >= target_k].copy()
chosen = k_candidates.sort_values(
    ["share_in_classes_lt_5", "age_width", "education_width"],
    ascending=[True, True, True],
).head(1)

if chosen.empty:
    chosen = settings_df.sort_values(["k_min", "share_unique"], ascending=[False, True]).head(1)

chosen


In [ ]:
chosen_params = chosen.iloc[0][["age_width", "education_width", "country", "marital"]].to_dict()
anon = generalize_adult(
    df,
    age_width=int(chosen_params["age_width"]),
    education_width=int(chosen_params["education_width"]),
    country=chosen_params["country"],
    marital=chosen_params["marital"],
)

print(chosen_params)
display(risk_report(anon, gen_qi).round(3))


## 7. Row Suppression

Suppression removes records that still belong to small equivalence classes. This can achieve a target `k`, but it changes the population being analyzed.


In [ ]:
def suppress_small_classes(data, qi_cols, k):
    sizes = equivalence_class_sizes(data, qi_cols).reset_index()
    keep_groups = sizes[sizes["equivalence_class_size"] >= k][qi_cols]
    kept = data.merge(keep_groups, on=qi_cols, how="inner")
    return kept.reset_index(drop=True)


suppressed = suppress_small_classes(anon, gen_qi, target_k)
print(f"Rows before suppression: {len(anon):,}")
print(f"Rows after suppression:  {len(suppressed):,}")
print(f"Retained share:          {len(suppressed) / len(anon):.3f}")
display(risk_report(suppressed, gen_qi).round(3))


### Exercise

Which groups are more likely to be suppressed? Why can this become a fairness or representativeness problem?



## 8. l-Diversity

k-anonymity only checks whether people are hidden in a group. l-diversity checks whether sensitive values vary inside that group.


In [ ]:
def l_diversity_report(data, qi_cols, sensitive_col):
    sensitive_counts = data.groupby(qi_cols, dropna=False)[sensitive_col].nunique().rename("distinct_sensitive_values")
    row_l = data.merge(sensitive_counts.reset_index(), on=qi_cols, how="left")["distinct_sensitive_values"]
    return pd.Series(
        {
            "min_l": int(sensitive_counts.min()),
            "share_rows_l_lt_2": float((row_l < 2).mean()),
            "share_groups_l_lt_2": float((sensitive_counts < 2).mean()),
        }
    )


display(l_diversity_report(suppressed, gen_qi, sensitive).round(3))


In [ ]:
bad_l_groups = (
    suppressed.groupby(gen_qi, dropna=False)
    .agg(n=(sensitive, "size"), distinct_income=(sensitive, "nunique"), income_values=(sensitive, lambda x: ", ".join(sorted(x.astype(str).unique()))))
    .query("distinct_income < 2")
    .sort_values("n", ascending=False)
)

bad_l_groups.head(10)


### Exercise

Pick one group with `distinct_income < 2`. Explain why an attacker can learn the sensitive value even though the release is k-anonymous.



## 9. t-Closeness

t-closeness compares the sensitive-value distribution inside each equivalence class with the global distribution.

For binary income, total variation distance reduces to the absolute difference between the group share of `>50K` and the global share of `>50K`.


In [ ]:
def t_closeness_report(data, qi_cols, sensitive_col, positive_value=">50K", t=0.15):
    global_positive = (data[sensitive_col] == positive_value).mean()

    group_stats = (
        data.assign(_positive=(data[sensitive_col] == positive_value).astype(int))
        .groupby(qi_cols, dropna=False)
        .agg(n=("_positive", "size"), positive_share=("_positive", "mean"))
        .reset_index()
    )
    group_stats["tv_distance"] = (group_stats["positive_share"] - global_positive).abs()
    row_distances = data.merge(group_stats[qi_cols + ["tv_distance"]], on=qi_cols, how="left")["tv_distance"]

    return pd.Series(
        {
            "global_positive_share": global_positive,
            "max_t_distance": group_stats["tv_distance"].max(),
            "share_rows_t_violation": (row_distances > t).mean(),
            "share_groups_t_violation": (group_stats["tv_distance"] > t).mean(),
        }
    ), group_stats.sort_values("tv_distance", ascending=False)


t_report, t_groups = t_closeness_report(suppressed, gen_qi, sensitive, t=0.15)
display(t_report.round(3))
display(t_groups.head(10).round(3))


### Exercise

Compare the worst t-closeness group with the global income distribution. Is this closer to a skewness problem, a homogeneity problem, or both?



## 10. Privacy-Utility Tradeoff

One way to measure utility is to ask whether a classifier trained on the privacy-transformed data still predicts income well. This is not the only notion of utility, but it is concrete.


In [ ]:
def make_model_frame(data, feature_cols, target_col):
    X = data[feature_cols].copy()
    y = (data[target_col] == ">50K").astype(int)
    return X, y


def evaluate_income_model(data, feature_cols, label):
    X, y = make_model_frame(data, feature_cols, sensitive)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
    )
    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = [c for c in X.columns if c not in numeric_features]

    try:
        one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

    model = Pipeline(
        steps=[
            (
                "preprocess",
                ColumnTransformer(
                    [
                        ("num", StandardScaler(), numeric_features),
                        ("cat", one_hot, categorical_features),
                    ]
                ),
            ),
            ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return {
        "dataset": label,
        "rows": len(data),
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
    }


original_features = ["age", "education_num", "marital_status", "race", "sex", "native_country", "hours_per_week", "workclass", "occupation"]
anonymous_features = gen_qi

utility_results = pd.DataFrame(
    [
        evaluate_income_model(df, original_features, "original detail"),
        evaluate_income_model(anon, anonymous_features, "generalized release"),
        evaluate_income_model(suppressed, anonymous_features, "generalized + suppressed"),
    ]
)

utility_results.round(3)


In [ ]:
privacy_utility = pd.DataFrame(
    [
        {"dataset": "original detail", **risk_report(df, base_qi).to_dict()},
        {"dataset": "generalized release", **risk_report(anon, gen_qi).to_dict()},
        {"dataset": "generalized + suppressed", **risk_report(suppressed, gen_qi).to_dict()},
    ]
).merge(utility_results, on="dataset")

display(privacy_utility[["dataset", "rows", "k_min", "share_unique", "share_in_classes_lt_5", "balanced_accuracy", "f1"]].round(3))


## 11. Differential Privacy for Aggregate Queries

Differential privacy protects the output of a randomized computation. For a count query, one person's record can change the count by at most 1, so the global sensitivity is 1.

The Laplace mechanism answers:

```text
private_answer = true_answer + Laplace(scale = sensitivity / epsilon)
```


In [ ]:
def laplace_mechanism(true_value, sensitivity, epsilon, random_state=None):
    local_rng = np.random.default_rng(random_state)
    scale = sensitivity / epsilon
    return true_value + local_rng.laplace(loc=0, scale=scale)


true_high_income_count = int((df[sensitive] == ">50K").sum())

epsilon_values = [0.1, 0.5, 1.0, 5.0]
dp_count_rows = []
for eps in epsilon_values:
    noisy_values = [
        laplace_mechanism(true_high_income_count, sensitivity=1, epsilon=eps, random_state=RANDOM_STATE + i)
        for i in range(200)
    ]
    dp_count_rows.append(
        {
            "epsilon": eps,
            "true_count": true_high_income_count,
            "noise_scale": 1 / eps,
            "mean_noisy_count": np.mean(noisy_values),
            "mean_absolute_error": np.mean(np.abs(np.array(noisy_values) - true_high_income_count)),
        }
    )

pd.DataFrame(dp_count_rows).round(2)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for eps in epsilon_values:
    noisy_values = [
        laplace_mechanism(true_high_income_count, sensitivity=1, epsilon=eps, random_state=10_000 + int(eps * 100) + i)
        for i in range(500)
    ]
    ax.hist(np.array(noisy_values) - true_high_income_count, bins=40, alpha=0.45, label=f"epsilon={eps}")

ax.set_title("Noise added to the high-income count")
ax.set_xlabel("noisy count - true count")
ax.set_ylabel("frequency")
ax.legend()
plt.show()


## 12. Sensitivity: Count, Sum, and Mean

Sensitivity depends on the query. For a bounded sum of `hours_per_week`, one record can change the sum by at most the upper bound. For a mean, we also need a minimum dataset size.


In [ ]:
hours = df["hours_per_week"].astype(float).clip(0, 80)

queries = pd.DataFrame(
    [
        {"query": "count high income", "true_value": true_high_income_count, "sensitivity": 1},
        {"query": "sum hours_per_week", "true_value": hours.sum(), "sensitivity": 80},
        {"query": "mean hours_per_week, n fixed", "true_value": hours.mean(), "sensitivity": 80 / len(hours)},
    ]
)
queries


In [ ]:
epsilon = 0.5
private_answers = queries.assign(
    epsilon=epsilon,
    noise_scale=queries["sensitivity"] / epsilon,
    one_private_answer=[
        laplace_mechanism(row.true_value, row.sensitivity, epsilon, RANDOM_STATE + i)
        for i, row in enumerate(queries.itertuples())
    ],
)

private_answers.round(3)


### Exercise

Why is the noisy sum much less accurate than the noisy count at the same `epsilon`? What bound did we impose to make the sum sensitivity finite?



## 13. Privacy Budget Composition

Repeated differentially private queries accumulate privacy loss. If each query spends part of the budget, the total privacy loss is the sum of the epsilons.


In [ ]:
budget_plan = pd.DataFrame(
    [
        {"query": "count records", "epsilon": 0.10},
        {"query": "count high-income records", "epsilon": 0.20},
        {"query": "mean hours_per_week", "epsilon": 0.20},
        {"query": "choose a model split", "epsilon": 0.30},
        {"query": "release final aggregate", "epsilon": 0.20},
    ]
)
budget_plan["cumulative_epsilon"] = budget_plan["epsilon"].cumsum()
budget_plan


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.step(range(1, len(budget_plan) + 1), budget_plan["cumulative_epsilon"], where="mid", marker="o")
ax.axhline(1.0, color="red", linestyle="--", label="budget = 1.0")
ax.set_xticks(range(1, len(budget_plan) + 1), budget_plan["query"], rotation=25, ha="right")
ax.set_ylabel("cumulative epsilon")
ax.set_title("Privacy budget accounting")
ax.legend()
plt.tight_layout()
plt.show()


## 14. Student Challenge

Choose one privacy goal and defend your design:

1. A 5-anonymous release with minimal row suppression.
2. A release where fewer than 10% of rows violate 2-diversity.
3. A release where fewer than 10% of rows violate t-closeness for `t = 0.15`.
4. A differentially private aggregate report with total budget `epsilon <= 1`.

For your chosen goal:

- Tune the generalization settings or DP budget allocation.
- Report at least one privacy metric.
- Report at least one utility metric.
- Explain what attack remains possible.
- Explain what information the analyst loses.


In [ ]:
# Challenge workspace.
# Try changing these parameters and re-running the reports above.
student_release = generalize_adult(
    df,
    age_width=20,
    education_width=8,
    country="region",
    marital="broad",
)
student_suppressed = suppress_small_classes(student_release, gen_qi, k=5)

display(risk_report(student_suppressed, gen_qi).round(3))
display(l_diversity_report(student_suppressed, gen_qi, sensitive).round(3))
display(t_closeness_report(student_suppressed, gen_qi, sensitive, t=0.15)[0].round(3))


## 15. Reflection

Answer briefly:

- Which method most directly reduces record linkage risk?
- Which method addresses homogeneous sensitive values?
- Which method addresses distribution shift between an equivalence class and the whole dataset?
- Which method protects query outputs without publishing microdata?
- Where did you observe the sharpest privacy-utility tradeoff?



## References

::: {#refs}
:::
